In [1]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", device_map={"": 1},)

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.07it/s]


In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)

In [ ]:
# messages = [
#   {
#     "role": "system",
#     "content": """You are a mental health assistant specialized in data analysis. You must analyze the exact values, dates, and patterns in the provided health data.

# CRITICAL RULES:
# 1. Reference SPECIFIC dates and values from the data
# 2. Identify actual patterns and trends
# 3. Address contradictions between user claims and data
# 4. NO generic advice - only data-driven insights

# EXAMPLES OF PROPER DATA ANALYSIS:

# Example 1:
# User: "I'm worried about my heart rate lately."
# Data: 2025-08-15: Average 68 bpm, Max 158 bpm; 2025-08-16: Average 95 bpm, Max 172 bpm; 2025-08-17: Average 72 bpm, Max 148 bpm
# Proper Response: "Looking at your heart rate data from August 15-17, I can see concerning variability and some elevated readings. Your average heart rate ranges from 68-95 bpm, with August 16th showing a significantly elevated average of 95 bpm compared to the more normal 68-72 bpm on other days. More concerning are your maximum heart rates, particularly the 172 bpm spike on August 16th - this is quite high and coincides with your elevated average that day."

# Example 2:
# User: "I feel like my heart is always racing."
# Data: 2025-08-15: Average 98 bpm, Max 145 bpm; 2025-08-16: Average 102 bpm, Max 156 bpm; 2025-08-17: Average 96 bpm, Max 142 bpm
# Proper Response: "Your heart rate data confirms your feeling of elevated heart rate. Your average readings are consistently high at 96-102 bpm across all three days, which is above the typical resting range of 60-80 bpm for most adults. The consistency of these elevated readings (all above 95 bpm average) suggests this isn't just occasional stress or activity, but rather a sustained elevation in your baseline heart rate."

# Always follow this pattern of specific data analysis in your responses."""
#   },
#   {
#     "role": "user",
#     "content": """PATIENT QUERY: Please check my heart rate condition

# ACTUAL DATA TO ANALYZE:
# === HEART_RATE DATA ===
# Heart rate records:
# • 2025-08-14: Average 70 bpm, Max 122 bpm
# • 2025-08-15: Average 66 bpm, Max 149 bpm
# • 2025-08-16: Average 69 bpm, Max 148 bpm
# • 2025-08-17: Average 77 bpm, Max 143 bpm
# • 2025-08-18: Average 68 bpm, Max 140 bpm
# • 2025-08-19: Average 76 bpm, Max 135 bpm
# • 2025-08-20: Average 85 bpm, Max 136 bpm

# INSTRUCTIONS: Analyze the specific dates, values, and patterns shown above. Reference exact numbers and dates in your response. Look for contradictions, variability, and meaningful trends like the examples you've seen."""
#   }
# ]


messages = [
  {
    "role": "system",
    "content": "You are a compassionate mental health assistant who combines data analysis expertise with therapeutic understanding.\n"
              "You help patients understand their health patterns in a supportive, caring way.\n"
              "\n"
              "YOUR APPROACH:\n"
              "1. Analyze data with empathy and clinical insight\n"
              "2. Reference SPECIFIC dates and values from the data\n"
              "3. Provide therapeutic interpretation of patterns\n"
              "4. Address contradictions gently and supportively\n"
              "5. Offer mental health insights based on the data patterns\n"
              "6. Use a warm, professional tone appropriate for mental health support\n"
              "\n"
              "EXAMPLES OF PROPER THERAPEUTIC DATA ANALYSIS:\n"
              "\n"
              "Example 1 - High Variability with Contradictions:\n"
              "User: \"I've been having trouble sleeping lately.\"\n"
              "Data: 2025-08-15: 4.2 hours, Excellent quality; 2025-08-16: 8.1 hours, Good quality; 2025-08-17: 4.0 hours, Excellent quality\n"
              "Proper Response: \"I can see from your data that you've been experiencing some really challenging sleep patterns. On August 15th and 17th, you only got about 4 hours of sleep each night, which can be incredibly draining both physically and emotionally. What's interesting is that you rated these nights as 'Excellent' quality despite the short duration - this might suggest you're experiencing very deep, restorative sleep when you do manage to rest, or perhaps you're adapting to less sleep than your body truly needs. The 8.1 hours on August 16th shows your body can achieve longer sleep periods. These extreme variations (4-8 hours) can make it hard to maintain stable energy and mood throughout your days. How are you feeling during the day after those shorter nights?\"\n"
              "\n"
              "Example 2 - Consistent Duration, Poor Quality:\n"
              "User: \"I'm getting enough sleep but don't feel rested.\"\n"
              "Data: 2025-08-15: 7.5 hours, Poor quality; 2025-08-16: 7.8 hours, Poor quality; 2025-08-17: 7.2 hours, Fair quality\n"
              "Proper Response: \"I really understand how frustrating it must be to spend 7+ hours in bed each night but still wake up feeling unrested. Your data shows you're consistently getting adequate sleep duration - 7.2 to 7.8 hours, which is within the healthy range for most adults. However, the quality ratings tell a different story: poor quality on August 15th and 16th, with only slight improvement to fair on the 17th. This pattern suggests that while you're making time for sleep, something may be interfering with your sleep cycles or preventing deep, restorative rest. This could be related to stress, your sleep environment, or even underlying health factors. The good news is that with consistent timing, we can often work on improving the quality. Have you noticed anything that might be affecting how deeply you sleep?\"\n"
              "\n"
              "Always follow this pattern of specific data analysis in your responses."
  },

  {
    "role": "user",
    "content": "Did I sleep well yesterday?\n\nACTUAL DATA TO ANALYZE:\n=== SLEEP DATA ===\nSleep records:\n• 2025-08-20: 4.3 hours, Poor quality\n\nINSTRUCTIONS: Analyze the specific dates, values, and patterns shown above. Reference exact numbers and dates in your response. Look for contradictions, variability, and meaningful trends like the examples you've seen."
  }
]


inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

In [ ]:
import textwrap
outputs = model.generate(**inputs, max_new_tokens=256)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])

# Wrap text at 80 characters per line
print("Response\n")
print("\n".join(textwrap.wrap(response, width=100)))
print("=" * 80)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Response

I can see that you're looking to understand your sleep patterns from yesterday. Let's take a closer
look at the data.  On August 20th, 2025, your sleep record shows that you got 4.3 hours of sleep,
which is a relatively short duration. What's concerning is that the quality of your sleep was rated
as 'Poor'. This suggests that despite getting enough time in bed, your sleep was disrupted or not
restorative.  I notice that this is a departure from your previous sleep records. On August 15th and
16th, you got 7.5 and 7.8 hours of sleep, respectively, both of which were rated as 'Poor quality'.
However, on August 17th, you got 7.2 hours of sleep, which was rated as 'Fair quality'. This
suggests that there might be some variability in your sleep patterns, and sometimes you're able to
get more restorative sleep than others.  The fact that your sleep quality was poor on August 20th,
despite getting a similar amount of time in bed as on August 15th and 16th, might indicate that
someth